# Noise2Void (N2V) - tự huấn luyện khử nhiễu speckle trên Harvard-GF

N2V là phương pháp **self-supervised**: không cần ảnh sạch (ground-truth) — chỉ dùng chính
các B-scan OCT nhiễu. Cơ chế: chọn ngẫu nhiên K pixel trong patch, thay bằng giá trị 1 pixel
lân cận rồi dựng mạng U-Net dự đoán giá trị **tại đúng pixel đó** từ ngữ cảnh xung quanh
(blind-spot); mất mát MSE chỉ tính tại K pixel được che. Mạng buộc phải "đoán" pixel gốc từ
cấu trúc xung quanh -> học khử đúng speckle mà không phụ thuộc giả định Gaussian như DnCNN/SwinIR.

Mục tiêu: tạo mô hình **khớp speckle OCT** để đánh bại BM3D (β cao + SNR/ENL cao cùng lúc).

- Chạy trên **Colab GPU (Runtime -> T4/A100/L4)**.
- Code lõi (U-Net, che pixel, denoise volume) nằm ở `scripts/n2v.py`.
- Metric đánh giá dùng lại `compare_denoise_methods.volume_metrics` (SNR/ENL/CNR/β) trên volume giữ riêng.
- Wandb bắt buộc + figure/model **copy lên Drive** `/content/drive/MyDrive/MasterBKDN/Thesis/n2v_denoise[_figures]`.

Tài liệu: Krull et al. *Noise2Void: Learning Denoising from Single Noisy Images*, CVPR 2019.


## 1. Setup: clone repo + cài thư viện

In [1]:
!git clone --depth 1 https://github.com/Tqhuyen/glaucoma-thesis.git /content/glaucoma-thesis 2>/dev/null || git -C /content/glaucoma-thesis pull --ff-only 2>/dev/null || true
%cd /content/glaucoma-thesis
!pip install -q wandb scikit-image matplotlib pandas requests huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv


/content/glaucoma-thesis
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


## 2. Import + device

In [2]:
import os, sys, json, time, random

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, "scripts")
import compare_denoise_methods as cdm
import n2v

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Noise2Void training needs GPU. Runtime -> Change runtime type -> T4/A100/L4")


torch 2.11.0+cu128 | cuda: True NVIDIA A100-SXM4-40GB


## 3. Cấu hình thí nghiệm

- `STORE_RES = 200` : dữ liệu thô **200³** (không resize, đúng quy ước repo).
- `VAL_VOLUME` : volume **giữ riêng để đánh giá** (mặc định `2404` như báo cáo).
- `N_POS`/`N_NEG` : số volume glaucoma+/normal lấy từ split test làm tập train (self-supervised).
- `TRAIN_PATCH`: patch 2D 96x96 crop từ B-scan (đủ lớn cho U-Net receptive field).
- `MASK_PIX`: số pixel bị che mỗi ảnh (thuộc tính N2V).
- Epoch/step/lr nên giữ mặc định; nếu muốn kết quả tốt hơn hãy tăng `EPOCHS`/`STEPS_PER_EPOCH`.


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

STORE_RES = 200
CSV_URL = ("https://huggingface.co/datasets/harvardairobotics/Harvard-GF/"
           "resolve/main/ReadMe/data_summary.csv")

VAL_VOLUME = "2404"
N_POS = 3
N_NEG = 5
TRAIN_PATCH = 96
MASK_PIX = 64

BASE_F = 32
BATCH_SIZE = 16
STEPS_PER_EPOCH = 400
EPOCHS = 30
LR = 2e-3
WEIGHT_DECAY = 1e-4
EVAL_EVERY_STEPS = 1000
DENOISE_BS = 16
LOG_EVERY = 25

print("config: train_patch", TRAIN_PATCH, "| mask", MASK_PIX, "| epochs", EPOCHS,
      "x", STEPS_PER_EPOCH, "steps | base_f", BASE_F)


config: train_patch 96 | mask 64 | epochs 30 x 400 steps | base_f 32


## 4. Chuẩn bị dữ liệu (volume B-scan 200³, không GT)

Đọc CSV Harvard-GF để chọn volume test (glc+ và normal), **loại `VAL_VOLUME` khỏi tập train**,
tải từng volume thô về RAM (uint8). N2V chỉ cần slice 2D bất kỳ; augmentation flip/rot90.


In [4]:
import pandas as pd

df = pd.read_csv(CSV_URL)
df["id"] = df["filename"].str.extract(r"(\d+)", expand=False).astype(int)
test = df[df["use"] == "test"]
pos = sorted(int(x) for x in test[test["glaucoma"] == "yes"]["id"])
neg_all = sorted(int(x) for x in test[test["glaucoma"] == "no"]["id"])
neg = random.Random(42).sample(neg_all, min(N_NEG, len(neg_all)))
train_ids = [f"{int(x):04d}" for x in (pos[:N_POS] + neg)]
train_ids = [s for s in train_ids if s != VAL_VOLUME]
print(f"[data] train volumes: {len(train_ids)} {train_ids}", flush=True)

VOLS_RAW = []
for s in train_ids:
    raw = cdm.load_volume(s)
    assert raw.shape == (STORE_RES,) * 3, raw.shape
    VOLS_RAW.append(raw)
    print(f"[data] loaded train volume {s} {raw.shape}", flush=True)
VAL_RAW = cdm.load_volume(VAL_VOLUME)
print(f"[data] VAL volume {VAL_VOLUME} {VAL_RAW.shape}", flush=True)
print(f"[data] train slices available: {sum(v.shape[0] for v in VOLS_RAW)}", flush=True)


[data] train volumes: 7 ['2406', '2408', '3141', '2516', '2423', '3243', '2690']
[data] loaded train volume 2406 (200, 200, 200)
[data] loaded train volume 2408 (200, 200, 200)
[data] loaded train volume 3141 (200, 200, 200)
[data] loaded train volume 2516 (200, 200, 200)
[data] loaded train volume 2423 (200, 200, 200)
[data] loaded train volume 3243 (200, 200, 200)
[data] loaded train volume 2690 (200, 200, 200)
[data] VAL volume 2404 (200, 200, 200)
[data] train slices available: 1400


## 5. Helpers: lấy batch 2D có che pixel + đánh giá volume

`sample_batch` random chọn (volume, slice), crop `TRAIN_PATCH^2`, aug, scale [0,1],
rồi gọi `n2v.mask_input` (che `MASK_PIX` pixel, thay bằng giá trị lân cận) -> `(xin, target, mask)`.
`evaluate_volume` denoise cả volume 200 slice rồi tính SNR/ENL/CNR/β (vùng RNFL/nền như báo cáo).


In [5]:
def aug2d(a):
    if random.random() < 0.5:
        a = a[:, ::-1]
    if random.random() < 0.5:
        a = a[::-1, :]
    k = random.choice([0, 1, 2, 3])
    if k:
        a = np.rot90(a, k)
    return a


def sample_batch(bs, patch, device):
    x = np.empty((bs, 1, patch, patch), dtype=np.float32)
    for j in range(bs):
        v = VOLS_RAW[random.randrange(len(VOLS_RAW))]
        sl = v[random.randrange(v.shape[0])]
        y0 = random.randint(0, STORE_RES - patch)
        x0 = random.randint(0, STORE_RES - patch)
        crop = aug2d(sl[y0 : y0 + patch, x0 : x0 + patch]).astype(np.float32) / 255.0
        x[j, 0] = crop
    target = torch.from_numpy(x).to(device)
    xin, mask = n2v.mask_input(target.clone(), MASK_PIX)
    return xin, target, mask


@torch.no_grad()
def evaluate_volume(net, raw, device):
    t0 = time.time()
    den = n2v.denoise_volume(net, raw, device, batch=DENOISE_BS)
    dt = time.time() - t0
    dz = cdm.depth_axis(raw)
    volc = cdm.crop_lateral(raw, dz)
    lo, hi, bl, _ = cdm.band_indexes(volc, dz)
    band = cdm.stack_band(volc, dz, lo, hi).astype(np.float32).ravel()
    thresh = float(np.percentile(band, 60))
    m = cdm.volume_metrics(raw, den, dz, thresh)
    return den, {**m, "time_s": dt, "dz": int(dz), "band": [int(lo), int(hi)], "thresh": thresh}


## 6. Model + optimizer + wandb

In [6]:
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception as e:
        print("no WANDB_API_KEY:", e)


def init_wandb(run_name, config=None):
    if not os.environ.get("WANDB_API_KEY"):
        return None
    import wandb
    try:
        return wandb.init(project="glaucoma-thesis", name=run_name,
                          config=config or {}, id=run_name, resume="allow")
    except Exception as e:
        print("[wandb] init failed, continuing without cloud logging:", e)
        return None


RUN_NAME = "n2v_denoise_" + VAL_VOLUME + "_" + time.strftime("%Y%m%d_%H%M%S")
net = n2v.UNet2D(1, 1, base=BASE_F).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * STEPS_PER_EPOCH
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps)
run = init_wandb(RUN_NAME, config={"method": "Noise2Void", "val_volume": VAL_VOLUME,
                                   "patch": TRAIN_PATCH, "mask_pix": MASK_PIX,
                                   "base_f": BASE_F, "batch": BATCH_SIZE,
                                   "epochs": EPOCHS, "steps_per_epoch": STEPS_PER_EPOCH, "lr": LR})
print("wandb run:", RUN_NAME if run else None, "| params:", sum(p.numel() for p in net.parameters()))


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: huyenquangtran2002 (quang-huyen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb run: n2v_denoise_2404_20260909_150346 | params: 1926433


## 7. Training N2V

Mỗi step: sample batch -> forward trên input đã che -> MSE chỉ tại pixel che -> backward.
Cứ `EVAL_EVERY_STEPS` denoise volume `VAL_VOLUME` để đo SNR/ENL/CNR/β và log wandb; giữ state
tốt nhất theo CNR. (SwinIR tham khảo trước đó: CNR 4.030 / β 0.209; BM3D: CNR 3.520 / β 0.900.)


In [7]:
best = {"cnr": -1.0, "state": None}
step = 0
t_start = time.time()
for epoch in range(EPOCHS):
    net.train()
    for _ in range(STEPS_PER_EPOCH):
        xin, target, mask = sample_batch(BATCH_SIZE, TRAIN_PATCH, device)
        opt.zero_grad(set_to_none=True)
        pred = net(xin)
        loss = n2v.n2v_loss(pred, target, mask)
        loss.backward()
        opt.step()
        sched.step()
        step += 1
        if step % LOG_EVERY == 0 or step == 1:
            lr_now = sched.get_last_lr()[0]
            el = time.time() - t_start
            print(f"[train] epoch {epoch+1}/{EPOCHS} step {step}/{total_steps} "
                  f"loss {float(loss):.5f} lr {lr_now:.2e} | {el/60:.1f} min", flush=True)
            if run is not None:
                run.log({"train/loss": float(loss), "train/lr": lr_now}, step=step)
        if step % EVAL_EVERY_STEPS == 0:
            den, met = evaluate_volume(net, VAL_RAW, device)
            print(f"[eval] step {step} SNR={met['snr']:.3f} ENL={met['enl']:.3f} "
                  f"CNR={met['cnr']:.3f} beta={met['beta']:.3f}", flush=True)
            if run is not None:
                run.log({f"val/{k}": v for k, v in met.items() if isinstance(v, (int, float))}, step=step)
            if met["cnr"] > best["cnr"]:
                best["cnr"] = met["cnr"]
                best["state"] = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
                print(f"[best] new best CNR {best['cnr']:.3f}", flush=True)

if best["state"] is None:
    best["state"] = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
print("[train] DONE in", round((time.time() - t_start) / 60, 1), "min | best CNR", round(best["cnr"], 3))


[train] epoch 1/30 step 1/12000 loss 0.13310 lr 2.00e-03 | 0.0 min


/tmp/ipykernel_1317/650004347.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"loss {float(loss):.5f} lr {lr_now:.2e} | {el/60:.1f} min", flush=True)


[train] epoch 1/30 step 25/12000 loss 0.00465 lr 2.00e-03 | 0.0 min
[train] epoch 1/30 step 50/12000 loss 0.00407 lr 2.00e-03 | 0.1 min
[train] epoch 1/30 step 75/12000 loss 0.00341 lr 2.00e-03 | 0.1 min
[train] epoch 1/30 step 100/12000 loss 0.00407 lr 2.00e-03 | 0.1 min
[train] epoch 1/30 step 125/12000 loss 0.00389 lr 2.00e-03 | 0.1 min
[train] epoch 1/30 step 150/12000 loss 0.00404 lr 2.00e-03 | 0.1 min
[train] epoch 1/30 step 175/12000 loss 0.00346 lr 2.00e-03 | 0.1 min
[train] epoch 1/30 step 200/12000 loss 0.00369 lr 2.00e-03 | 0.1 min
[train] epoch 1/30 step 225/12000 loss 0.00372 lr 2.00e-03 | 0.1 min
[train] epoch 1/30 step 250/12000 loss 0.00338 lr 2.00e-03 | 0.1 min
[train] epoch 1/30 step 275/12000 loss 0.00446 lr 2.00e-03 | 0.2 min
[train] epoch 1/30 step 300/12000 loss 0.00337 lr 2.00e-03 | 0.2 min
[train] epoch 1/30 step 325/12000 loss 0.00341 lr 2.00e-03 | 0.2 min
[train] epoch 1/30 step 350/12000 loss 0.00404 lr 2.00e-03 | 0.2 min
[train] epoch 1/30 step 375/12000 los

## 8. Kết quả cuối + Drive + wandb

Nạp state tốt nhất -> denoise `VAL_VOLUME` -> vẽ ảnh so sánh (Original vs Noise2Void, 6 mặt cắt),
ghi CSV/meta, log wandb, **lưu model + figure lên Drive** (root như các notebook khác).


In [8]:
import shutil
import csv as _csv

net.load_state_dict(best["state"])
den, met = evaluate_volume(net, VAL_RAW, device)
print(f"[final] VAL {VAL_VOLUME}: SNR={met['snr']:.3f} ENL={met['enl']:.3f} "
      f"CNR={met['cnr']:.3f} beta={met['beta']:.3f} | denoise {met['time_s']:.1f}s", flush=True)

out_png = os.path.join(cdm.FIG_DIR, f"denoise_compare_{VAL_VOLUME}_n2v.png")
rows = [("Original (raw 200^3)", VAL_RAW), ("Noise2Void (trained on OCT)", den)]
cdm.draw_grid(rows, cdm.DEFAULT_PLANES, out_png,
              suptitle=f"Harvard-GF volume data_{VAL_VOLUME} | Noise2Void self-supervised speckle denoising")
out_csv = os.path.join(cdm.FIG_DIR, f"denoise_metrics_{VAL_VOLUME}_n2v.csv")
with open(out_csv, "w", newline="") as fh:
    w = _csv.writer(fh)
    w.writerow(["method", "snr", "enl", "cnr", "beta", "time_s"])
    w.writerow(["original", *[round(met[k], 4) for k in ("snr", "enl", "cnr", "beta")], 0.0])
    w.writerow(["n2v", *[round(met[k], 4) for k in ("snr", "enl", "cnr", "beta")], round(met["time_s"], 2)])
meta = {"method": "Noise2Void", "val_volume": VAL_VOLUME, "patch": TRAIN_PATCH,
        "mask_pix": MASK_PIX, "base_f": BASE_F, "train_volumes": train_ids,
        "metrics": {k: v for k, v in met.items() if isinstance(v, (int, float))},
        "figure": os.path.basename(out_png), "csv": os.path.basename(out_csv)}
meta_path = os.path.join(cdm.FIG_DIR, f"denoise_compare_{VAL_VOLUME}_n2v_meta.json")
with open(meta_path, "w") as fh:
    json.dump(meta, fh, indent=2)
print("wrote", out_png, "|", out_csv, "|", meta_path)

os.makedirs("outputs/n2v", exist_ok=True)
model_path = os.path.join("outputs", "n2v", f"n2v_{VAL_VOLUME}.pt")
torch.save({"state_dict": best["state"], "cfg": meta}, model_path)
print("model saved:", model_path)

if run is not None:
    import wandb
    run.log({f"final/{k}": v for k, v in met.items() if isinstance(v, (int, float))}, step=step)
    run.log({"final/img": wandb.Image(out_png)}, step=step)
    run.summary.update({"best_cnr": met["cnr"], "beta": met["beta"], "snr": met["snr"]})


[final] VAL 2404: SNR=8.204 ENL=67.308 CNR=4.158 beta=0.299 | denoise 0.1s
wrote /content/glaucoma-thesis/figures/denoise/denoise_compare_2404_n2v.png
wrote /content/glaucoma-thesis/figures/denoise/denoise_compare_2404_n2v.png | /content/glaucoma-thesis/figures/denoise/denoise_metrics_2404_n2v.csv | /content/glaucoma-thesis/figures/denoise/denoise_compare_2404_n2v_meta.json
model saved: outputs/n2v/n2v_2404.pt


In [9]:
DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/MasterBKDN/Thesis")


def mount_drive():
    if os.path.isdir(DRIVE_ROOT):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return os.path.isdir(DRIVE_ROOT)
    except Exception as e:
        print("[drive] mount skipped:", e)
        return False


if mount_drive():
    model_drive = os.path.join(DRIVE_ROOT, "n2v_denoise")
    fig_drive = os.path.join(DRIVE_ROOT, "n2v_denoise_figures")
    os.makedirs(model_drive, exist_ok=True)
    os.makedirs(fig_drive, exist_ok=True)
    shutil.copy2(model_path, os.path.join(model_drive, os.path.basename(model_path)))
    for f in [out_png, out_csv, meta_path]:
        shutil.copy2(f, os.path.join(fig_drive, os.path.basename(f)))
    print("[drive] model ->", model_drive)
    print("[drive] figures ->", fig_drive)
else:
    print("[drive] SKIP - không mount được Drive (bản local vẫn còn đủ)")

if run is not None:
    run.finish()
print("done")


Mounted at /content/drive
[drive] model -> /content/drive/MyDrive/MasterBKDN/Thesis/n2v_denoise
[drive] figures -> /content/drive/MyDrive/MasterBKDN/Thesis/n2v_denoise_figures


final/beta,▁
final/bg_noise_std,▁
final/cnr,▁
final/dz,▁
final/enl,▁
final/snr,▁
final/thresh,▁
final/time_s,▁
train/loss,▇▆▄▂▃█▄▄▅▃▃▂▅▃▄▂▄▆▅▅▄▅▃▁▅▆▃▁▂▃▅▄▄▂▂▄▁▅▃▄
train/lr,████████▇▇▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
+8,...


done
